# Essai « mouvement » : images d'époque → petites vidéos

Carnet du site *Histoire du XIXe siècle* (spec 16). **Voie normale : la CLI** (`python3 scripts/diaporamas/generation.py lancer <slug>`, script `mouvement.py`) ; ce carnet est la voie manuelle de secours. Modèle à poids ouverts **Wan 2.2 TI2V-5B** (Apache 2.0), image → vidéo de 5 s à 720p.

**Mode d'emploi**

1. Menu *Exécution* › *Modifier le type d'exécution* › GPU **L4** (ou **A100** s'il est proposé). Un T4 ne suffit pas.
2. *Exécution* › *Tout exécuter*.
3. Quand la cellule « Kit » le demande, choisir le fichier `kit-mouvement.zip` (dans `.cache/diaporamas/revolution/generation/` du projet).
4. Attendre : téléchargement du modèle ≈ 5 min, puis ≈ 5 à 20 min par clip selon la carte. Le zip des résultats (`resultats-mouvement.zip`) se télécharge tout seul à la fin.
5. Sur le Mac : `python3 scripts/diaporamas/generation.py encoder revolution ~/Downloads/resultats-mouvement.zip`, puis ouvrir la page de visionnage indiquée.

Si la session s'arrête en route, relancer *Tout exécuter* : les clips déjà faits et encore dans `/content/sortie` ne sont pas refaits (ils sont perdus si Colab a recyclé la machine ; `DRIVE = True` les copie aussi sur Drive).

In [ ]:
# Réglages
PROFIL = "qualite"      # "qualite" : 1280×704, 121 images (5 s), 40 pas ; "rapide" : 832×480, 81 images (3,4 s), 25 pas
ESSAI_RAPIDE = True     # commence par un clip « rapide » de la première image pour valider la chaîne (≈ 3 à 6 min)
DRIVE = False           # copie aussi les résultats dans Mon Drive/19eme-siecle/generation/<slug>/ (demande une autorisation)
SEULEMENT = []          # ex. [18, 56] : ne faire que ces diapositives ; [] = toutes

In [ ]:
# Carte graphique
import torch
assert torch.cuda.is_available(), "Pas de GPU : Exécution › Modifier le type d'exécution › GPU"
nom_gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 2**30
print(nom_gpu, f"{vram:.0f} Go")
if vram < 20:
    raise SystemExit("Carte trop petite (T4 ?) : choisir L4 ou A100 dans le type d'exécution.")

In [ ]:
# Bibliothèques (≈ 2 min ; diffusers pris sur la branche principale, comme le recommande la fiche du modèle ; torchao retiré : la version préinstallée sur Colab casse diffusers)
%pip uninstall -y -q torchao
%pip install -q -U transformers accelerate ftfy imageio imageio-ffmpeg sentencepiece protobuf
%pip install -q git+https://github.com/huggingface/diffusers
import diffusers, transformers
print("diffusers", diffusers.__version__, "· transformers", transformers.__version__)

In [ ]:
# Kit : manifeste + images sources (envoyer kit-mouvement.zip)
import json, pathlib, zipfile
from google.colab import files
kit = pathlib.Path("/content/kit")
if not (kit / "manifeste.json").exists():
    envoye = files.upload()
    nom_zip = next(iter(envoye))
    zipfile.ZipFile(nom_zip).extractall(kit)
manifeste = json.loads((kit / "manifeste.json").read_text(encoding="utf-8"))
print(manifeste["slug"], "·", len(manifeste["elements"]), "éléments")
for el in manifeste["elements"]:
    print(" ", el["id"], el["largeur"], "×", el["hauteur"], "·", el["titre"][:60])

In [ ]:
# Modèle (≈ 22 Go à télécharger la première fois dans la session)
import torch
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
MODELE = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"

def charger(**types):
    vae = AutoencoderKLWan.from_pretrained(MODELE, subfolder="vae", **{k: torch.float32 for k in types})
    return WanImageToVideoPipeline.from_pretrained(MODELE, vae=vae, **{k: torch.bfloat16 for k in types})

try:
    pipe = charger(torch_dtype=None)
except TypeError:
    pipe = charger(dtype=None)        # diffusers récents : l'argument s'appelle dtype
if vram < 40:
    pipe.enable_model_cpu_offload()   # L4 24 Go : un composant à la fois sur la carte
    if hasattr(pipe.vae, "enable_tiling"):
        pipe.vae.enable_tiling()
else:
    pipe.to("cuda")                    # A100 : tout sur la carte
mod = pipe.vae_scale_factor_spatial * pipe.transformer.config.patch_size[1]
print("prêt · dimensions multiples de", mod, "px")

In [ ]:
# Génération
import json, math, time, pathlib
from PIL import Image
from diffusers.utils import export_to_video

PROFILS = {
    "qualite": dict(aire=1280 * 704, images=121, pas=40),
    "rapide":  dict(aire=832 * 480,  images=81,  pas=25),
}
sortie = pathlib.Path("/content/sortie"); sortie.mkdir(exist_ok=True)
journal = sortie / "journal.jsonl"

def dimensions(im, aire):
    r = im.height / im.width
    h = max(mod, round(math.sqrt(aire * r)) // mod * mod)
    w = max(mod, round(math.sqrt(aire / r)) // mod * mod)
    return w, h

def generer(el, profil, suffixe=""):
    p = PROFILS[profil]
    f = sortie / f"{el['id']}{suffixe}.mp4"
    if f.exists():
        print("déjà fait :", f.name); return
    im = Image.open(kit / el["source"]).convert("RGB")
    w, h = dimensions(im, p["aire"])
    im = im.resize((w, h), Image.LANCZOS)
    g = torch.Generator(device="cuda").manual_seed(int(el.get("graine", 0)))
    t0 = time.time()
    print(f"{f.name} : {w}×{h}, {p['images']} images, {p['pas']} pas…", flush=True)
    frames = pipe(
        image=im, prompt=el["prompt"], negative_prompt=manifeste["negatif"],
        height=h, width=w, num_frames=p["images"], guidance_scale=5.0,
        num_inference_steps=p["pas"], generator=g,
    ).frames[0]
    export_to_video(frames, str(f), fps=24)
    d = time.time() - t0
    ligne = dict(id=el["id"], profil=profil, fichier=f.name, largeur=w, hauteur=h,
                 images=p["images"], pas=p["pas"], secondes=round(d), gpu=nom_gpu)
    with journal.open("a", encoding="utf-8") as j:
        j.write(json.dumps(ligne, ensure_ascii=False) + "\n")
    print(f"  → {d / 60:.1f} min")

elements = [e for e in manifeste["elements"] if not SEULEMENT or e["n"] in SEULEMENT]
if ESSAI_RAPIDE and elements:
    generer(elements[0], "rapide", "-rapide")
for el in elements:
    generer(el, PROFIL)
print("terminé :", sorted(f.name for f in sortie.glob("*.mp4")))

In [ ]:
# Résultats : zip téléchargé (et copie Drive si demandée)
import shutil, pathlib
from google.colab import files
zip_sortie = shutil.make_archive("/content/resultats-mouvement", "zip", sortie)
if DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = pathlib.Path("/content/drive/MyDrive/19eme-siecle/generation") / manifeste["slug"]
    dest.mkdir(parents=True, exist_ok=True)
    for f in sortie.iterdir():
        shutil.copy(f, dest / f.name)
    print("copié dans", dest)
files.download(zip_sortie)